# Simulación de Códigos LDPC

Este cuaderno implementa un código LDPC sencillo y el decodificador de
propagación de creencias (Belief Propagation, BP), ilustrando el artículo
[Códigos LDPC y Turbo](../../02-teoria-informacion/11-codigos-ldpc-y-turbo.md).

Usamos solo la biblioteca estándar de Python.

In [ ]:
import random
import math
from collections import defaultdict

## Utilidades básicas

In [ ]:
def gf2_multiply(H, x):
    """Multiplica H*x en GF(2). H es lista de listas, x es lista de bits."""
    m = len(H)
    n = len(x)
    result = [0] * m
    for i in range(m):
        s = 0
        for j in range(n):
            s ^= H[i][j] * x[j]
        result[i] = s
    return result

def check_syndrome(H, codeword):
    """Devuelve True si H*codeword == 0 (mod 2)."""
    return all(s == 0 for s in gf2_multiply(H, codeword))

def bsc_channel(bits, p_flip):
    """Paso por canal BSC con probabilidad de volteo p_flip."""
    return [b ^ (1 if random.random() < p_flip else 0) for b in bits]

def bit_error_rate(a, b):
    """Fracción de posiciones donde a y b difieren."""
    return sum(x != y for x, y in zip(a, b)) / len(a)

## Ejemplo 1: Código de repetición (3,1)

El código de repetición más sencillo: el bit se repite 3 veces.
Matriz de paridad H con 2 filas: x1 XOR x2 = 0 y x2 XOR x3 = 0.

In [ ]:
# Código (3,1): H tiene 2 ecuaciones de paridad sobre 3 bits
H_rep = [
    [1, 1, 0],  # x1 + x2 = 0
    [0, 1, 1],  # x2 + x3 = 0
]

# Palabras código válidas: 000 y 111
codewords = [[0,0,0], [1,1,1]]
for cw in codewords:
    s = gf2_multiply(H_rep, cw)
    print(f"codeword {cw}, síndrome {s}, válido: {all(si==0 for si in s)}")

# Palabra con error en posición 0
received = [1, 0, 0]
s = gf2_multiply(H_rep, received)
print(f"\nRecibido {received}, síndrome {s} → hay error")

## Ejemplo 2: Grafo de Tanner

Representamos el grafo bipartito del código LDPC:
- Nodos variable: v0, v1, ..., v_{n-1}
- Nodos restricción: c0, c1, ..., c_{m-1}
- Arista (vi, cj) si H[j][i] == 1

In [ ]:
def build_tanner_graph(H):
    """Construye listas de adyacencia del grafo de Tanner.
    
    Retorna:
        var_neighbors[i]: restricciones conectadas al nodo variable i
        check_neighbors[j]: variables conectadas al nodo restricción j
    """
    m = len(H)
    n = len(H[0])
    var_neighbors = [[] for _ in range(n)]
    check_neighbors = [[] for _ in range(m)]
    for j in range(m):
        for i in range(n):
            if H[j][i] == 1:
                var_neighbors[i].append(j)
                check_neighbors[j].append(i)
    return var_neighbors, check_neighbors

# Código LDPC (3,6)-regular pequeño: n=6, m=3, tasa=1/2
# Cada variable conectada a dv=2 restricciones, cada restricción a dc=4 variables
H_ldpc = [
    [1, 1, 0, 1, 0, 0],
    [0, 1, 1, 0, 1, 0],
    [1, 0, 1, 0, 0, 1],
]

var_nb, check_nb = build_tanner_graph(H_ldpc)
print("Grafo de Tanner:")
for i, nb in enumerate(var_nb):
    print(f"  Variable v{i} conectada a restricciones: {nb}")
for j, nb in enumerate(check_nb):
    print(f"  Restricción c{j} conectada a variables: {nb}")

## Ejemplo 3: Decodificador Belief Propagation (sum-product)

El algoritmo BP intercambia mensajes en LLR (Log-Likelihood Ratio):
- LLR positivo → bit más probablemente 0
- LLR negativo → bit más probablemente 1

In [ ]:
def bp_decode(H, llr_channel, max_iters=50):
    """Decodificador Belief Propagation en LLR para canal BSC/AWGN.
    
    Args:
        H: matriz de paridad
        llr_channel: LLR inicial desde el canal para cada bit
        max_iters: número máximo de iteraciones BP
    
    Returns:
        (decision, converged): bits decodificados y si convergió
    """
    m = len(H)
    n = len(H[0])
    var_nb, check_nb = build_tanner_graph(H)
    
    # Mensajes variable→restricción y restricción→variable, indexados por (var, check)
    msg_v2c = defaultdict(float)  # mensaje LLR de variable i a restricción j
    msg_c2v = defaultdict(float)  # mensaje LLR de restricción j a variable i
    
    # Inicialización: mensaje variable→restricción = LLR del canal
    for i in range(n):
        for j in var_nb[i]:
            msg_v2c[(i, j)] = llr_channel[i]
    
    for iteration in range(max_iters):
        # Actualización restricción→variable
        # Fórmula: L_{c→v} = 2 arctanh( prod_{v'≠v} tanh(L_{v'→c}/2) )
        for j in range(m):
            for i in check_nb[j]:
                prod = 1.0
                for ip in check_nb[j]:
                    if ip != i:
                        x = msg_v2c[(ip, j)] / 2.0
                        # Clamp para evitar overflow
                        x = max(-20.0, min(20.0, x))
                        prod *= math.tanh(x)
                # Clamp prod para evitar domain error en atanh
                prod = max(-1 + 1e-10, min(1 - 1e-10, prod))
                msg_c2v[(j, i)] = 2.0 * math.atanh(prod)
        
        # Actualización variable→restricción y LLR total
        llr_total = [0.0] * n
        for i in range(n):
            llr_total[i] = llr_channel[i] + sum(msg_c2v[(j, i)] for j in var_nb[i])
            for j in var_nb[i]:
                # Mensaje a j excluye el mensaje entrante de j
                msg_v2c[(i, j)] = llr_channel[i] + sum(
                    msg_c2v[(jp, i)] for jp in var_nb[i] if jp != j
                )
        
        # Decisión hard
        decision = [0 if llr > 0 else 1 for llr in llr_total]
        
        # Comprobar convergencia
        if check_syndrome(H, decision):
            return decision, True
    
    return decision, False


def llr_bsc(received_bits, p_flip):
    """LLR para canal BSC: log(P(y|x=0)/P(y|x=1))."""
    llr_0 = math.log((1 - p_flip) / p_flip)
    llr_1 = math.log(p_flip / (1 - p_flip))
    return [llr_0 if b == 0 else llr_1 for b in received_bits]


print("Decodificador BP definido.")

## Ejemplo 4: Simulación de BER vs. probabilidad de volteo

In [ ]:
def simulate_ber(H, p_flip, n_trials=200, max_iters=20):
    """Simula la tasa de error de bit (BER) del decodificador BP."""
    n = len(H[0])
    total_bit_errors = 0
    total_bits = 0
    
    for _ in range(n_trials):
        # Codeword todo-ceros (siempre válida para código lineal)
        sent = [0] * n
        received = bsc_channel(sent, p_flip)
        llr = llr_bsc(received, p_flip)
        decoded, converged = bp_decode(H, llr, max_iters)
        total_bit_errors += sum(d != s for d, s in zip(decoded, sent))
        total_bits += n
    
    return total_bit_errors / total_bits


# Código LDPC pequeño (n=6, m=3)
p_values = [0.05, 0.08, 0.10, 0.12, 0.15, 0.18, 0.20]

print("BER del decodificador BP vs. probabilidad de volteo BSC")
print(f"{'p_flip':>8} | {'BER sin cod.':>12} | {'BER con BP':>10}")
print("-" * 38)

random.seed(42)
for p in p_values:
    ber = simulate_ber(H_ldpc, p, n_trials=300)
    print(f"{p:>8.2f} | {p:>12.4f} | {ber:>10.4f}")

## Ejemplo 5: Ciclos en el grafo de Tanner

Los ciclos cortos (longitud 4) degradan el decodificador BP porque crean
correlaciones entre mensajes. Verificamos si H_ldpc tiene ciclos de longitud 4.

In [ ]:
def find_cycles_length_4(H):
    """Detecta ciclos de longitud 4 en el grafo de Tanner.
    
    Un ciclo de longitud 4 existe si dos nodos restricción
    comparten dos o más nodos variable.
    """
    m = len(H)
    n = len(H[0])
    _, check_nb = build_tanner_graph(H)
    
    cycles = []
    for j1 in range(m):
        for j2 in range(j1 + 1, m):
            shared = set(check_nb[j1]) & set(check_nb[j2])
            if len(shared) >= 2:
                shared_list = sorted(shared)
                cycles.append((j1, j2, shared_list[:2]))
    return cycles


cycles = find_cycles_length_4(H_ldpc)
if cycles:
    print("Ciclos de longitud 4 encontrados:")
    for j1, j2, (v1, v2) in cycles:
        print(f"  c{j1} - v{v1} - c{j2} - v{v2} - c{j1}")
else:
    print("No hay ciclos de longitud 4: grafo libre de ciclos cortos.")

# Matriz sin ciclos de longitud 4
H_sin_ciclos = [
    [1, 1, 0, 0, 1, 0, 0, 1],
    [0, 1, 1, 0, 0, 1, 0, 0],
    [0, 0, 1, 1, 0, 0, 1, 0],
    [1, 0, 0, 1, 0, 0, 0, 1],
]
cycles2 = find_cycles_length_4(H_sin_ciclos)
print(f"\nH_sin_ciclos: {len(cycles2)} ciclos de longitud 4")

## Ideas clave

- Los códigos LDPC tienen matrices de paridad **escasas** (baja densidad), lo que
  permite decodificación eficiente en tiempo O(n·iteraciones).
- El grafo de **Tanner** visualiza la estructura del código como grafo bipartito;
  los ciclos cortos degradan la propagación de creencias.
- El decodificador **BP en LLR** intercambia mensajes de log-razón de verosimilitud;
  los LLR permiten trabajar con sumas en vez de productos.
- La tasa de error cae bruscamente al cruzar el **umbral** de decodificación BP.

## Ejercicios

1. Verifica que la tasa del código `H_ldpc` (n=6, m=3) es R=k/n=3/6=1/2.
2. Implementa una función que genere una matriz LDPC (dv, dc)-regular aleatoria
   sin ciclos de longitud 4 usando el algoritmo de Gallager.
3. Compara la BER del código de repetición (3,1) con el código LDPC (6,3)
   a igual tasa. ¿Cuál es mejor?
4. Modifica `bp_decode` para registrar el número de iteraciones hasta convergencia
   y traza el histograma para p=0.05 y p=0.10.